# Notebook 03 — GMM Regime Analysis

Deep-dive into GMM-based market regime detection.

Key insight: a single full-sample VaR estimate averages across regimes
and consistently underestimates tail risk during stress. Per-regime VaR
can be 3-5x larger in the stressed regime than in the calm regime.


In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import yfinance as yf
import warnings
warnings.filterwarnings('ignore')

from src.regime_detection import detect_market_regimes, regime_var_summary
from src.var_models import historical_var

print('Libraries loaded.')

In [ ]:
# SPY + QQQ + IWM 2018-2024
data = yf.download(['SPY', 'QQQ', 'IWM'], start='2018-01-01', end='2024-01-01', progress=False)['Close']
returns = data.pct_change().dropna()
print(f'Returns: {len(returns)} observations')

## 1. GMM Regime Detection

In [ ]:
N_REGIMES = 3
regime_df, port_returns = detect_market_regimes(returns, n_regimes=N_REGIMES)
print(f'Detected {N_REGIMES} regimes over {len(regime_df)} observations')
print('\nRegime counts:')
print(regime_df['regime'].value_counts().sort_index())

In [ ]:
# Per-regime statistics
summary = regime_var_summary(port_returns, regime_df, N_REGIMES, confidence_level=0.05)
print('\nPer-Regime VaR Summary (95% confidence):')
display_df = summary.copy()
for col in ['Ann Return', 'Ann Volatility', 'VaR (historical)']:
    display_df[col] = display_df[col].map(lambda x: f'{x:.2%}')
display_df['Freq (%)'] = display_df['Freq (%)'].map(lambda x: f'{x:.1f}%')
print(display_df.to_string(index=False))

# Full-sample VaR for comparison
full_var = historical_var(port_returns, 0.05)
print(f'\nFull-sample 95% VaR: {full_var:.3%}')
print('Compare to per-regime values above — the stressed-regime VaR is the realistic measure.')

## 2. Regime Timeline

In [ ]:
colors = ['#2196F3', '#F44336', '#4CAF50', '#FF9800']
regime_names = ['Low-vol Bull', 'High-vol Stress', 'Recovery/Sideways', 'Regime 3']

fig = make_subplots(rows=2, cols=1,
                     subplot_titles=['Portfolio Returns by Regime', 'Regime Probability Over Time'],
                     shared_xaxes=True)

for r in range(N_REGIMES):
    mask = regime_df['regime'] == r
    regime_rets = port_returns[mask]
    fig.add_trace(
        go.Scatter(x=regime_rets.index, y=regime_rets * 100, mode='markers',
                   name=regime_names[r], marker=dict(color=colors[r], size=3)),
        row=1, col=1
    )

for r in range(N_REGIMES):
    fig.add_trace(
        go.Scatter(x=regime_df.index, y=regime_df[f'prob_regime_{r}'],
                   name=f'P(Regime {r})', line=dict(color=colors[r])),
        row=2, col=1
    )

fig.update_layout(height=700, title='GMM Market Regime Detection — SPY/QQQ/IWM 2018-2024')
fig.update_yaxes(title_text='Return (%)', row=1, col=1)
fig.update_yaxes(title_text='Probability', row=2, col=1)
fig.show()

## 3. VaR by Regime — The Regime Risk Premium

In [ ]:
regime_vars = {}
for r in range(N_REGIMES):
    mask = regime_df['regime'] == r
    regime_rets = port_returns[mask]
    if len(regime_rets) > 20:
        regime_vars[f'Regime {r}'] = {
            '95% VaR': historical_var(regime_rets, 0.05),
            '99% VaR': historical_var(regime_rets, 0.01),
        }

regime_var_df = pd.DataFrame(regime_vars).T * 100
regime_var_df['Full Sample'] = [full_var * 100, historical_var(port_returns, 0.01) * 100]
print('VaR by Regime vs Full-Sample (%)')
print(regime_var_df.round(3).to_string())

vars_95 = [v['95% VaR'] * 100 for v in regime_vars.values()]
fig = go.Figure(data=[
    go.Bar(x=list(regime_vars.keys()), y=[-v for v in vars_95],
           marker_color=colors[:len(vars_95)],
           text=[f'{abs(v):.2f}%' for v in vars_95], textposition='auto')
])
fig.add_hline(y=-full_var * 100, line_dash='dash', line_color='black',
               annotation_text=f'Full-sample VaR: {-full_var*100:.2f}%')
fig.update_layout(title='95% VaR by Market Regime', yaxis_title='VaR (loss %)', height=400)
fig.show()